In [1]:
import nest_asyncio
nest_asyncio.apply()

In [1]:
import sys
import os
import asyncio
import logging

# lightrag
sys.path.append(
    os.path.abspath("..")
)

from src.lightrag.rag import *

In [17]:
rag = await initialize_rag()

DEBUG: Process 37060 Shared-Data already initialized (multiprocess=False)
DEBUG: Captured embedding max_token_size: 8192
DEBUG: LightRAG init with param:
  working_dir = c:\Users\erwin\lightRAG_Experiment\py310_lightRAG\penilaian_makalah\notebooks\rag_storage_documents,
  kv_storage = PGKVStorage,
  vector_storage = PGVectorStorage,
  graph_storage = Neo4JStorage,
  doc_status_storage = JsonDocStatusStorage,
  workspace = ,
  log_level = None,
  log_file_path = None,
  top_k = 40,
  chunk_top_k = 20,
  max_entity_tokens = 6000,
  max_relation_tokens = 8000,
  max_total_tokens = 30000,
  cosine_threshold = 0.2,
  related_chunk_number = 5,
  kg_chunk_pick_method = VECTOR,
  entity_extract_max_gleaning = 1,
  max_extract_input_tokens = 20480,
  force_llm_summary_on_merge = 8,
  chunk_token_size = 1200,
  chunk_overlap_token_size = 100,
  tokenizer = <lightrag.utils.TiktokenTokenizer object at 0x0000025399D8EF80>,
  tiktoken_model_name = gpt-4o-mini,
  chunking_func = <function chunking_by

Initializing storages...


INFO: [base] Connected to lightrag-baseline at neo4j://127.0.0.1:7687
INFO: [base] Ensured B-Tree index on entity_id for base in lightrag-baseline
INFO: [base] Found existing index 'entity_id_fulltext_idx_base' with state: ONLINE
INFO: [base] Full-text index 'entity_id_fulltext_idx_base' already exists and is online. Skipping recreation.
DEBUG: Process 37060 storage namespace already initialized: [doc_status]
DEBUG: All storage types initialized


In [18]:
# Test embedding function
test_text = ["This is a test string for embedding."]
embedding = await rag.embedding_func(test_text)
embedding_dim = embedding.shape[1]
print("\n=======================")
print("Test embedding function")
print("========================")
print(f"Test dict: {test_text}")
print(f"Detected embedding dimension: {embedding_dim}\n\n")

INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)



Test embedding function
Test dict: ['This is a test string for embedding.']
Detected embedding dimension: 1536




### Prompts


In [19]:
from src.prompt.prompts import QUERY_KEYWORDS, PROMPT_KONTEKS
from lightrag import QueryParam

context_response = await rag.aquery(
    query=QUERY_KEYWORDS.format(selected_jabatan="Sekretaris Utama"),
    param=QueryParam(
        only_need_context=True, 
        mode="hybrid",
        user_prompt=PROMPT_KONTEKS,
        # enable_rerank=False,
        # max_token_for_context=2000,
    ),
)

context_response_llm = await rag.aquery(
    query=QUERY_KEYWORDS.format(selected_jabatan="Sekretaris Utama"),
    param=QueryParam(
        only_need_context=False, 
        mode="hybrid",
        user_prompt=PROMPT_KONTEKS,
    ), 
)


DEBUG: [aquery_llm] Query param: QueryParam(mode='hybrid', only_need_context=True, only_need_prompt=False, response_type='Multiple Paragraphs', stream=False, top_k=40, chunk_top_k=20, max_entity_tokens=6000, max_relation_tokens=8000, max_total_tokens=30000, hl_keywords=[], ll_keywords=[], conversation_history=[], history_turns=0, model_func=None, user_prompt="\n---Role---\nAnda adalah asisten persiapan konteks yang bertugas merangkum informasi relevan\ndari knowledge graph untuk membantu evaluator menilai makalah seleksi.\n \n---Goal---\nBerdasarkan context yang diberikan dari knowledge graph SKJ BPOM, susun ringkasan\nkontekstual yang terstruktur untuk jabatan '{selected_jabatan}' guna mendukung\npenilaian makalah seleksi kompetensi bidang.\n \n---Batasan Penting---\n- Gunakan HANYA informasi yang tersedia dalam context yang diberikan.\n- Jangan menambahkan informasi dari pengetahuan umum Anda di luar context.\n- Jika informasi tertentu tidak tersedia dalam context, nyatakan secara ek

In [20]:
# n2
print(context_response)


Knowledge Graph Data (Entity):

```json
{"entity": "Penilaian Kompetensi Teknis", "type": "concept", "description": "Penilaian Kompetensi Teknis meliputi aspek penulisan makalah, presentasi, dan wawancara untuk menilai kemampuan peserta."}
{"entity": "BPOM", "type": "organization", "description": "Badan Pengawas Obat dan Makanan (BPOM) adalah lembaga pemerintah di Indonesia yang memiliki tanggung jawab utama dalam pengawasan dan regulasi obat dan makanan. BPOM bertujuan untuk memastikan kesehatan dan keamanan produk yang beredar di masyarakat melalui pengawasan yang ketat terhadap obat-obatan dan makanan. Lembaga ini berfokus pada pelayanan publik yang berkaitan dengan keselamatan warga negara dan pelaksanaan standar keamanan yang berlaku.\n\nSebagai badan pengawas, BPOM tidak hanya mengawasi penggunaan obat dan makanan, tetapi juga terlibat dalam pengembangan kompetensi dan peningkatan mutu produk. BPOM melaksanakan fungsi pengawasan dan penindakan terhadap peraturan di bidang obat d

In [21]:
print(context_response)


Knowledge Graph Data (Entity):

```json
{"entity": "Penilaian Kompetensi Teknis", "type": "concept", "description": "Penilaian Kompetensi Teknis meliputi aspek penulisan makalah, presentasi, dan wawancara untuk menilai kemampuan peserta."}
{"entity": "BPOM", "type": "organization", "description": "Badan Pengawas Obat dan Makanan (BPOM) adalah lembaga pemerintah di Indonesia yang memiliki tanggung jawab utama dalam pengawasan dan regulasi obat dan makanan. BPOM bertujuan untuk memastikan kesehatan dan keamanan produk yang beredar di masyarakat melalui pengawasan yang ketat terhadap obat-obatan dan makanan. Lembaga ini berfokus pada pelayanan publik yang berkaitan dengan keselamatan warga negara dan pelaksanaan standar keamanan yang berlaku.\n\nSebagai badan pengawas, BPOM tidak hanya mengawasi penggunaan obat dan makanan, tetapi juga terlibat dalam pengembangan kompetensi dan peningkatan mutu produk. BPOM melaksanakan fungsi pengawasan dan penindakan terhadap peraturan di bidang obat d

In [22]:
print(context_response_llm)

## 1. Profil Jabatan
Jabatan **Sekretaris Utama** pada Badan Pengawas Obat dan Makanan (BPOM) berfungsi sebagai koordinasi pelaksanaan tugas, pembinaan, dan dukungan administrasi kepada seluruh unit organisasi. Ia juga bertanggung jawab dalam penyusunan rencana, program, anggaran, serta pengelolaan kegiatan keprotokolan dan kesekretariatan.

## 2. Kompetensi yang Dipersyaratkan
### Kompetensi Teknis
- **Kemampuan dalam Manajemen Perkantoran**: Mengelola administrasi, kearsipan, dan protokol.
- **Penyusunan Laporan**: Penyusunan laporan pimpinan dan pengelola data penting untuk pengambilan keputusan.

### Kompetensi Manajerial
- **Integritas**: Mampu menciptakan situasi kerja yang mendorong kepatuhan pada nilai dan etika organisasi.
- **Kerjasama**: Membangun tim yang efektif dan sinergis antar unit kerja.
- **Komunikasi**: Mampu menyampaikan informasi dengan jelas dan persuasif.
- **Orientasi pada Hasil**: Memastikan unit kerja mencapai target yang ditetapkan.
- **Pelayanan Publik**: M

In [23]:
dataset = []

dataset.append(
    {
        "user_input":QUERY_KEYWORDS,
        "retrieved_contexts": [context_response],
        "response":context_response_llm,
    } 
)


In [24]:
print(dataset)

[{'user_input': '\nStandar Kompetensi Jabatan {selected_jabatan}\nKompetensi Teknis {selected_jabatan}\nKompetensi Manajerial {selected_jabatan}\nKompetensi Sosial Kultural {selected_jabatan}\nIndikator perilaku kompetensi {selected_jabatan}\nPersyaratan jabatan {selected_jabatan}\nTugas pokok dan fungsi {selected_jabatan}\nPenilaian Penulisan Makalah\nForm. 1 Penilaian Penulisan Makalah\nKriteria penilaian makalah seleksi kompetensi bidang\nKetajaman analisis kompetensi bidang\nSistematika penulisan makalah JPT\nKesesuaian isi makalah dengan tema jabatan\nPEMETAAN VISI MISI TUJUAN STRATEGI SASARAN BPOM\nPEMETAAN ARAH KEBIJAKAN DAN STRATEGI BPOM\nMATRIKS RINGKASAN ANALISIS SWOT BPOM\nRencana Strategis BPOM\n', 'retrieved_contexts': ['\nKnowledge Graph Data (Entity):\n\n```json\n{"entity": "Penilaian Kompetensi Teknis", "type": "concept", "description": "Penilaian Kompetensi Teknis meliputi aspek penulisan makalah, presentasi, dan wawancara untuk menilai kemampuan peserta."}\n{"entity":

In [25]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset)

In [26]:
evaluation_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response'], len=1)

In [27]:
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL", "gpt-4-mini"),
    api_key=os.getenv("LLM_BINDING_API_KEY"),
    base_url=os.getenv("LLM_BINDING_HOST"),
    temperature=0.1,
)

LLM_PENILAI = ChatOpenAI (
    model = os.getenv("LLM_PENILAI", "anthropic/claude-sonnet-4.6"),
    api_key=os.getenv("LLM_BINDING_API_KEY"),
    base_url=os.getenv("LLM_BINDING_HOST"),
    temperature=1.0,
) 

evaluator_llm = LangchainLLMWrapper(llm)

C:\Users\erwin\AppData\Local\Temp\ipykernel_37060\282288279.py:18: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(llm)


In [28]:
from ragas import evaluate 
from ragas.metrics import Faithfulness

result = evaluate(
        dataset=evaluation_dataset,
        metrics=[Faithfulness()],
        llm=evaluator_llm,
    )
    

C:\Users\erwin\AppData\Local\Temp\ipykernel_37060\4131673675.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness
Evaluating: 100%|██████████| 1/1 [01:09<00:00, 69.55s/it]


In [29]:
result

{'faithfulness': 1.0000}

### Test penilaian 

In [30]:
# Muindar
from src.helper.helper import _read_pdf
makalah_path = "../data/Makalah Uji/Makalah Teknis Manajemen Talenta_Muindar_2026 ok.pdf"


makalah_text = _read_pdf(makalah_path)
print(makalah_text)


MAKALAH
Strategi Balai Besar POM untuk Memperkuat Pengawasan Obat dan
Makanan Berbasis Risiko yang Adaptif, Responsif, dan Terpadu di Masa
Efisiensi, dalam Rangka Mendukung Asta Cita Pemerintah, Meningkatkan
Daya Saing dan Perekonomian Melalui Pemanfaatan Teknologi Informasi
dan Digital
MUINDAR
BBPOM di Gorontalo
I. PENDAHULUAN
1. Latar Belakang
Pengawasan terhadap obat dan makanan adalah salah satu fungsi
strategis negara untuk melindungi kesehatan masyarakat dan juga
mendukung perekonomian nasional. Obat dan makanan yang aman,
berkualitas, dan berkhasiat tidak hanya melindungi konsumen, tetapi
juga meningkatkan kepercayaan masyarakat terhadap regulasi dan daya
saing industri nasional. Sistem pengawasan yang efektif sangat krusial
untuk menciptakan ekosistem industri yang sehat dan berkelanjutan
dalam konteks pembangunan nasional.
Seluruh produk yang beredar di Indonesia harus memenuhi standar
keamanan, mutu, dan manfaat. Oleh karena itu, Badan Pengawas Obat
dan Makanan sebagai lembag

In [16]:
makalah_dummy = """
BAB 1  
PENDAHULUAN 
1.1 Latar Belakang Masalah  
PT. Mercedes-Benz Indonesia adalah sebuah perusahaan yang bergerak 
dibidang otomotif asal Jerman yang memproduksi berbagai macam kendaraan 
seperti mobil, truk, dan bus. Mercedes-Benz adalah sebuah merek mobil dari 
perusahaan Daimler Chrysler (dulunya dikenal sebagai Daimler-Benz), yang 
dikenal umum dengan nama Mercedes. Perusahaan pembuat mobil Mercedes-Benz 
ini adalah perusahaan mobil tertua di dunia yang sekarang menjadi produsen mobil 
mewah dalam "German Big 3" bersama dengan Audi dan BMW yang menghasilkan 
mobil-mobil mewah terbaik di dunia. Kantor Mercedes-Benz Jakarta berlokasi di 
Deutsche Bank Building, sedangkan pabrik PT. Mercedes-Benz Indonesia berdiri 
di areal seluas 42 hektar yang terletak di Desa Wanaherang, Gunung Putri Bogor 
16965 Indonesia.  
Sistem produksi merupakan serangkaian aktivitas yang dilakukan untuk 
mengolah atau mengubah sejumlah masukan (input) menjadi sejumlah keluaran 
(output) yang memiliki nilai tambah. Pengolahan yang terjadi bisa secara fisik 
maupun nonfisik. Setiap industri manufaktur pasti memiliki yang namanya 
engineering internalnya masing-masing, memang tidak terlalu berpengaruh besar 
dalam proses industri manufaktur namun sangat berpengaruh besar terhadap 
kelancaran dari proses manufakturing. 
Proses yang dilakukan sistem produksi bagian engineering PT. Mercedes-Benz 
Indonesia dalam hal monitoring masih menggunakan cara manual. Karena cara 
kerja sistem yang berjalan di sistem produksi tersebut masih bersifat manual 
sehingga timbul masalah yang terjadi, yaitu para supervisor harus berkeliling di 
area line untuk memantau dan mengetahui masalah yang terjadi disetiap station. 
Begitu juga bagi para operator disetiap station yang mengalami masalah dan harus 
berkeliling untuk mencari supervisor. 
1 
Dengan permasalahan tersebut dapat diatasi dengan mengubah sistem 
monitoring tersebut. Melihat penelitian sebelumnya yang dilakukan pada bidang 
industri farmasi oleh Ayu Anugrah Rizki, Dewi Shofi,dan Iyan Bachtiar[1], lalu 
pada bidang manufaktur industri logam dasar dan elektronika oleh Mariana 
Yulistiana Lubis[2], dan bidang industri pembuatan sepatu oleh Ruhimat Fauzia 
Akbar, M. Ary Murti, dan M. Ramdhani[3] yang menerapkan sistem andon Dari 
ketiga penelitian tersebut menyebutkan bahwa sistem andon diterapkan untuk 
mengatasi masalah pada sistem monitoring pada produksi. Dengan permasalah 
yang sama, pada sistem produksi bagian engineering PT. Mercedes-Benz Indonesia 
maka diputuskan untuk menerapkan sistem andon yang akan dituangkan ke dalam 
laporan penelitian yang berjudul “PROTOTYPE ANDON SISTEM PADA 
PT.MERCEDES-BENZ INDONESIA”. 
1.2 Rumusan Masalah 
Beberapa rumusan masalah berdasarkan latar belakang, adalah sebagai berikut: 
1. Bagaimana merancang sebuah sistem agar supervisor dapat memantau 
setiap station dan menyelesaikan masalah yang terjadi pada line?  
2. Bagaimana merancang sebuah sistem agar supervisor dapat memanggil 
bagian Quality, Maintenance, dan Production untuk meminta bantuan 
menyelesaikan masalah yang terjadi pada station? 
1.3 Maksud dan Tujuan 
Penelitian ini dimaksudkan sebagai konsep penerapan sistem andon dalam 
memonitoring sistem produksi pada bagian engineering, yang bertujuan: 
1. Membatu supervisor memantau setiap station yang digunakan untuk 
menyelesaikan masalah yang terjadi pada line. 
2. Memudahkan proses memanggil bagian Quality, Maintenance, dan 
Production untuk meminta bantuan menyelesaikan masalah yang terjadi 
pada station. 
2 
1.4 Manfaat 
Manfaat praktis yang diperoleh dari penerapan sistem ini yaitu :  
1. Bagi supervisor, penerapan sistem andon ini diharapkan dapat 
meningkatkan dan memudahkan kinerja supervisor dalam memonitoring 
setiap station pada line. 
2. Bagi penulis, dengan penerapan sistem andon ini penulis jadi paham betapa 
pentingnya sistem monitoring, terutama di perusahaan besar yang 
permintaan produksinya tinggi. 
1.5 Batasan Masalah 
Dalam penulisan laporan terdapat batasan masalah dalam ruang lingkup 
sebagai berikut: 
1. Pembangunan sistem menggunakan metode prototype. 
2. Sistem dapat mengolah data dari PPC(Perencanaan & Pengendalian 
Produksi) dan menghasilkan data informasi produksi yang realtime. 
3. Data yang diolah: 
a. Data produksi dari PPC(Perencanaan & Pengendalian Produksi). 
4. Output yang dihasilkan sistem berupa display data informasi produksi. 
1.6 Metodelogi Penelitian 
Dalam penelitian ini menggunakan beberapa metodologi yaitu:  
1. Studi Literatur  
Mempelajari konsep dasar yang berhubungan dengan prinsip 
kerjaandon system,  mikrokontroler dan komunikasi serial. 
2. Studi Lapangan  
Mempelajari proses perakitan yang ada dibagian engineer di 
PT.Mercedes-Benz, beserta menumpulkan data-data yang diperlukan 
guna dijadikan sebagai model pembuatan dan penerapan sistem andon. 
3. Perancangan prototype andon system 
3 
Membuat rancangan sistem andon beserta display menggunakan 
aplikasi Adobe XD. Perancangan ini disesuaikan dengan kebutuhan dan 
kondisi dilapangan. 
4. Pengujian Hasil Rancangan 
Menguji keberhasilan prototype sistem andon, dengan pengujian yang 
dilakukan pada proses pengiriman sinyal informasi sampai dengan 
munculnya report pada display di PC. Selain itu juga dilakukan 
pengujian pada ferformansi sistem andon terhadap proses produksi. 
1.7  Sistematika Penulisan  
Sistematika yang digunakan penulis akan memuat uraian secara garis besar dari 
isi penelitian dalam tiap bab, yaitu sebagai berikut: 
BAB I PENDAHULUAN 
Bab ini berisi penjelasan tentang latar belakang, perumusan masalah, maksud dan 
tujuan, manfaat, batasan masalah, metodologi penelitian dan sistematika penulisan. 
BAB 2 TINJAUAN PUSTAKA 
Bab ini memberikan informasi tentang PT.Mercedes-Benz serta landasan teori yang 
berkaitan dengan permasalahan yang ada dalam penelitian. 
BAB 3 PEMBAHASAN 
Bab ini membahas tentang hasil dari pemecahan masalah berdasarkan analisis 
masalah, analisis sistem yang sedang berjalan, analisis kebutuhan perangkat keras, 
serta perancangan sistem, implementasi, dan pengujian. 
BAB 4 KESIMPULAN DAN SARAN 
Bab ini merupakan bab terakhir yang berisi kesimpulan yang didapatkan dari 
pembahasan dan saran untuk penelitian selanjutnya.
"""

In [ ]:
from src.prompt.prompts import PROMPT_PENILAIAN

evaluation_prompt = PROMPT_PENILAIAN.format(
    assessment_context=context_response_llm,
    makalah_text=makalah_dummy,
    tema_text="""Strategi Balai Besar POM Dalam Meningkatkan Perkuatan Pengawasan 
Obat dan Makanan Berbasis Resiko yang Adatif, Responsive dan Terintegrasi 
di era efisiensi untuk mendukung Asta Cita Pemerintah, daya saing dan 
peningkatan perekonomian dengan memanfaatakan Teknologi Informasi 
Digitalisasi.""", 
    # tema_text=tema_text or "Tidak ada tema khusus",
)

eval_response = LLM_PENILAI.invoke(evaluation_prompt)
eval_response


AIMessage(content='```json\n{\n  "Ringkasan": "Makalah ini membahas perancangan prototype sistem andon pada PT. Mercedes-Benz Indonesia, sebuah perusahaan otomotif asal Jerman. Makalah mengangkat permasalahan sistem monitoring manual pada bagian engineering produksi, di mana supervisor harus berkeliling secara fisik untuk memantau setiap station dan operator harus mencari supervisor secara manual. Solusi yang diusulkan adalah penerapan sistem andon berbasis teknologi untuk memonitoring proses produksi secara real-time. Makalah ini sama sekali tidak berkaitan dengan tema yang ditetapkan, yaitu strategi Balai Besar POM dalam meningkatkan pengawasan obat dan makanan berbasis risiko yang adaptif, responsif, dan terintegrasi, serta tidak relevan dengan jabatan Sekretaris Utama BPOM.",\n\n  "scores": {\n    "n1": 40,\n    "n2": 40,\n    "n3": 55,\n    "n4": 40,\n    "n5": 60\n  },\n\n  "final_score": 46\n}\n```', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'comple

In [72]:
# Eval response data dummy tidak relevan (mercedez benz)
print(eval_response)

content='```json\n{\n  "Ringkasan": "Makalah ini membahas perancangan prototype sistem andon pada PT. Mercedes-Benz Indonesia, sebuah perusahaan otomotif asal Jerman. Makalah mengangkat permasalahan sistem monitoring manual pada bagian engineering produksi, di mana supervisor harus berkeliling secara fisik untuk memantau setiap station dan operator harus mencari supervisor secara manual. Solusi yang diusulkan adalah penerapan sistem andon berbasis teknologi untuk memonitoring proses produksi secara real-time. Makalah ini sama sekali tidak berkaitan dengan tema yang ditetapkan, yaitu strategi Balai Besar POM dalam meningkatkan pengawasan obat dan makanan berbasis risiko yang adaptif, responsif, dan terintegrasi, serta tidak relevan dengan jabatan Sekretaris Utama BPOM.",\n\n  "scores": {\n    "n1": 40,\n    "n2": 40,\n    "n3": 55,\n    "n4": 40,\n    "n5": 60\n  },\n\n  "final_score": 46\n}\n```' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens'

In [31]:
from src.prompt.prompts import PROMPT_PENILAIAN

evaluation_prompt = PROMPT_PENILAIAN.format(
    assessment_context=context_response_llm,
    makalah_text=makalah_text,
    tema_text="""Strategi Balai Besar POM Dalam Meningkatkan Perkuatan Pengawasan 
Obat dan Makanan Berbasis Resiko yang Adatif, Responsive dan Terintegrasi 
di era efisiensi untuk mendukung Asta Cita Pemerintah, daya saing dan 
peningkatan perekonomian dengan memanfaatakan Teknologi Informasi 
Digitalisasi.""", 
    # tema_text=tema_text or "Tidak ada tema khusus",
)

eval_response = LLM_PENILAI.invoke(evaluation_prompt)


In [32]:
# Makalah asli 2
print(eval_response)

content='```json\n{\n  "Ringkasan": "Makalah ini ditulis oleh Muindar dari BBPOM di Gorontalo dengan tema strategi penguatan pengawasan obat dan makanan berbasis risiko yang adaptif, responsif, dan terintegrasi di era efisiensi anggaran. Makalah membahas peran Balai Besar POM sebagai unit pelaksana teknis dalam mengimplementasikan kebijakan pengawasan di daerah, serta relevansinya dengan program Asta Cita pemerintahan Prabowo Subianto, khususnya Asta Cita 4 dan beberapa Asta Cita lainnya. Struktur makalah terdiri dari pendahuluan yang memuat latar belakang, rumusan masalah, tujuan, dan manfaat; pembahasan yang mencakup analisis konsep melalui matriks SWOT, dampak strategi, dan rencana aksi (Plan of Action) jangka pendek, menengah, dan panjang; serta penutup berisi kesimpulan. Analisis SWOT mengidentifikasi kekuatan seperti regulasi yang kuat dan jaringan kerja sama luas, kelemahan seperti keterbatasan SDM dan sistem TI yang belum terintegrasi, peluang berupa pertumbuhan pasar dan trans

In [ ]:
# Makalah asli 2
print(eval_response)

content='```json\n{\n  "Ringkasan": "Makalah ini ditulis oleh Muindar dari BBPOM di Gorontalo dengan judul yang selaras dengan tema yang ditetapkan. Makalah membahas strategi Balai Besar POM dalam memperkuat pengawasan obat dan makanan berbasis risiko yang adaptif, responsif, dan terintegrasi di era efisiensi, dalam rangka mendukung Asta Cita Pemerintah melalui pemanfaatan teknologi informasi dan digitalisasi. Struktur makalah terdiri dari pendahuluan (latar belakang, rumusan masalah, tujuan, dan manfaat), pembahasan (analisis konsep menggunakan matriks SWOT, dampak, dan plan of action), serta penutup dan daftar pustaka. Analisis strategis menggunakan pendekatan SWOT mengidentifikasi kekuatan seperti regulasi yang kuat dan jaringan kerja sama, kelemahan seperti keterbatasan SDM dan sistem TI yang belum terintegrasi, peluang seperti pertumbuhan pasar dan transformasi digital, serta ancaman seperti kejahatan siber dan maraknya hoaks. Strategi yang dirumuskan mencakup penguatan pengawasan

### Test uncertainty 

### Muindar

In [34]:
print(evaluation_prompt)


---Role---

Anda adalah evaluator akademik sebagai Panitia Seleksi yang bertugas menilai kualitas substansi makalah secara objektif dan sistematis. Penilaian harus didasarkan hanya pada isi makalah yang tersedia, dengan mempertimbangkan konteks jabatan yang dituju.

---Goal---

Melakukan penilaian terhadap makalah berdasarkan kriteria penilaian yang telah ditentukan, memberikan skor numerik untuk setiap kriteria.

---Konteks Jabatan---

## 1. Profil Jabatan
Jabatan **Sekretaris Utama** pada Badan Pengawas Obat dan Makanan (BPOM) berfungsi sebagai koordinasi pelaksanaan tugas, pembinaan, dan dukungan administrasi kepada seluruh unit organisasi. Ia juga bertanggung jawab dalam penyusunan rencana, program, anggaran, serta pengelolaan kegiatan keprotokolan dan kesekretariatan.

## 2. Kompetensi yang Dipersyaratkan
### Kompetensi Teknis
- **Kemampuan dalam Manajemen Perkantoran**: Mengelola administrasi, kearsipan, dan protokol.
- **Penyusunan Laporan**: Penyusunan laporan pimpinan dan pen

In [17]:
### Run ke 2 muindar 05 temp

### Sample M = 7 

In [36]:
import json
import re
import asyncio
import numpy as np
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# --- KONFIGURASI DARI FEEDBACK ---
M_SAMPLES = 7 # Menggunakan M=5 sudah cukup (Minimum viable) dan menghemat biaya token
MIN_SCORE = 40
MAX_SCORE = 100
SCORE_RANGE = MAX_SCORE - MIN_SCORE
NSV_THRESHOLD = 0.1 # Threshold 10% dari range (setara std dev > 6 poin)

llm_sampler = ChatOpenAI(
    model=os.getenv("LLM_PENILAI", "anthropic/claude-sonnet-4.6"),
    api_key=os.getenv("LLM_BINDING_API_KEY"),
    base_url=os.getenv("LLM_BINDING_HOST"),
    temperature=0.5
)

async def get_sample_score(prompt_text):
    """Fungsi mengambil sampel dengan filter validasi (outlier handling)"""
    try:
        response = await llm_sampler.ainvoke([HumanMessage(content=prompt_text)])
        content = response.content.strip()
        
        # 2. Logika RegEx untuk mengekstrak teks di dalam bracket { ... }
        # Ini akan mengabaikan basa-basi Claude atau markdown ```json
        json_match = re.search(r'\{.*\}', content, re.DOTALL)
        
        if not json_match:
            print(f"⚠️ Gagal menemukan format JSON di output model.")
            return None
            
        clean_json_string = json_match.group(0)
        data = json.loads(clean_json_string)
        
        # Ekstraksi dan Validasi rentang skor
        scores = data.get("scores", {})
        valid_scores = {}
        for k, v in scores.items():
            if isinstance(v, (int, float)) and MIN_SCORE <= v <= MAX_SCORE:
                valid_scores[k] = float(v)
                
        return valid_scores if valid_scores else None
    except Exception as e:
        return None

async def calculate_score_uncertainty(prompt_text):
    print(f"🚀 Memulai {M_SAMPLES} sampling untuk ekstraksi ketidakpastian skor...")
    
    # Menjalankan sampling paralel
    tasks = [get_sample_score(prompt_text) for _ in range(M_SAMPLES)]
    results = await asyncio.gather(*tasks)
    
    # Struktur data penampung
    score_distributions = {
        "n1": [],
        "n2": [],
        "n3": [], 
        "n4": [],
        "n5": []
    }
    
    # Memasukkan skor tervalidasi ke dalam distribusi
    for res in results:
        if not res: continue
        for key in score_distributions.keys():
            if key in res:
                score_distributions[key].append(res[key])
                
    # --- KALKULASI METRIK UNCERTAINTY ---
    evaluation_results = {
        "consensus_scores": {},
        "uncertainty": {"per_criteria": {}}
    }
    
    nsv_dict = {}
    
    for criteria, values in score_distributions.items():
        if not values: continue
        
        mean_score = np.mean(values)
        std_dev = np.std(values, ddof=1) if len(values) > 1 else 0.0
        
        # Metrik 1: NSV (Normalized Score Variance)
        nsv = std_dev / SCORE_RANGE
        nsv_dict[criteria] = nsv
        
        status = "⚠️ PERLU REVIEW" if nsv > NSV_THRESHOLD else "✅ YAKIN"
        
        evaluation_results["consensus_scores"][criteria] = round(mean_score, 2)
        evaluation_results["uncertainty"]["per_criteria"][criteria] = {
            "nsv": round(nsv, 3),
            "std": round(std_dev, 2),
            "status": status,
            "raw_samples": values
        }
        
    # Metrik 2: WAU (Weighted Aggregate Uncertainty)
    # Menghitung agregat dengan bobot n4 dikalikan 2 (Total pembagi 6)
    if all(k in nsv_dict for k in score_distributions.keys()):
        wau = (
            nsv_dict["n1"] + 
            nsv_dict["n2"] + 
            nsv_dict["n3"] + 
            (2 * nsv_dict["n4"]) + 
            nsv_dict["n5"]
        ) / 6
        
        evaluation_results["uncertainty"]["weighted_aggregate"] = round(wau, 3)
        evaluation_results["uncertainty"]["overall_status"] = "⚠️ BUTUH REVIEW HUMAN" if wau > NSV_THRESHOLD else "✅ YAKIN (Konsisten)"
        
        # Mencari kriteria dengan ketidakpastian tertinggi
        evaluation_results["uncertainty"]["most_uncertain_criteria"] = max(nsv_dict, key=nsv_dict.get)

    # --- TAMPILKAN HASIL ---
    print("\n" + "="*60)
    print("📊 HASIL ANALISIS KETIDAKPASTIAN SKOR MAKALAH (NSV & WAU)")
    print("="*60)
    print(json.dumps(evaluation_results, indent=2))
    print("="*60)

# Eksekusi fungsi dengan passing variabel prompt secara eksplisit
# Asumsi: variabel evaluation_prompt sudah berisi prompt lengkap
await calculate_score_uncertainty(evaluation_prompt)

🚀 Memulai 7 sampling untuk ekstraksi ketidakpastian skor...

📊 HASIL ANALISIS KETIDAKPASTIAN SKOR MAKALAH (NSV & WAU)
{
  "consensus_scores": {
    "n1": 80.29,
    "n2": 74.14,
    "n3": 69.29,
    "n4": 65.0,
    "n5": 72.29
  },
  "uncertainty": {
    "per_criteria": {
      "n1": {
        "nsv": 0.036,
        "std": 2.14,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          78.0,
          78.0,
          82.0,
          82.0,
          78.0,
          82.0,
          82.0
        ]
      },
      "n2": {
        "nsv": 0.038,
        "std": 2.27,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          72.0,
          72.0,
          75.0,
          75.0,
          72.0,
          78.0,
          75.0
        ]
      },
      "n3": {
        "nsv": 0.037,
        "std": 2.21,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          70.0,
          65.0,
          70.0,
          70.0,
          68.0,
          72.0,
          70.0
        

#### t = 1.0 

In [ ]:
M_SAMPLES = 7
MIN_SCORE = 40
MAX_SCORE = 100
SCORE_RANGE = MAX_SCORE - MIN_SCORE
NSV_THRESHOLD = 0.1 # Threshold 10% dari range (setara std dev > 6 poin)

llm_sampler = ChatOpenAI(
    model=os.getenv("LLM_PENILAI", "anthropic/claude-sonnet-4.6"),
    api_key=os.getenv("LLM_BINDING_API_KEY"),
    base_url=os.getenv("LLM_BINDING_HOST"),
    temperature=1.0
)


await calculate_score_uncertainty(evaluation_prompt)

🚀 Memulai 7 sampling untuk ekstraksi ketidakpastian skor...

📊 HASIL ANALISIS KETIDAKPASTIAN SKOR MAKALAH (NSV & WAU)
{
  "consensus_scores": {
    "n1": 80.29,
    "n2": 74.14,
    "n3": 69.0,
    "n4": 65.0,
    "n5": 71.71
  },
  "uncertainty": {
    "per_criteria": {
      "n1": {
        "nsv": 0.036,
        "std": 2.14,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          78.0,
          82.0,
          82.0,
          82.0,
          78.0,
          78.0,
          82.0
        ]
      },
      "n2": {
        "nsv": 0.038,
        "std": 2.27,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          72.0,
          75.0,
          75.0,
          75.0,
          72.0,
          72.0,
          78.0
        ]
      },
      "n3": {
        "nsv": 0.037,
        "std": 2.24,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          65.0,
          72.0,
          70.0,
          70.0,
          68.0,
          68.0,
          70.0
        ]

#### t= 1.5


In [38]:
M_SAMPLES = 7
MIN_SCORE = 40
MAX_SCORE = 100
SCORE_RANGE = MAX_SCORE - MIN_SCORE
NSV_THRESHOLD = 0.1 # Threshold 10% dari range (setara std dev > 6 poin)

llm_sampler = ChatOpenAI(
    model=os.getenv("LLM_PENILAI", "anthropic/claude-sonnet-4.6"),
    api_key=os.getenv("LLM_BINDING_API_KEY"),
    base_url=os.getenv("LLM_BINDING_HOST"),
    temperature=1.5
)


await calculate_score_uncertainty(evaluation_prompt)

🚀 Memulai 7 sampling untuk ekstraksi ketidakpastian skor...

📊 HASIL ANALISIS KETIDAKPASTIAN SKOR MAKALAH (NSV & WAU)
{
  "consensus_scores": {
    "n1": 80.86,
    "n2": 73.86,
    "n3": 70.0,
    "n4": 64.57,
    "n5": 72.86
  },
  "uncertainty": {
    "per_criteria": {
      "n1": {
        "nsv": 0.033,
        "std": 1.95,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          78.0,
          82.0,
          78.0,
          82.0,
          82.0,
          82.0,
          82.0
        ]
      },
      "n2": {
        "nsv": 0.03,
        "std": 1.77,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          72.0,
          76.0,
          72.0,
          72.0,
          75.0,
          75.0,
          75.0
        ]
      },
      "n3": {
        "nsv": 0.027,
        "std": 1.63,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          68.0,
          72.0,
          68.0,
          70.0,
          70.0,
          72.0,
          70.0
        ]

### Sample M = 5 

In [18]:
import json
import re
import asyncio
import numpy as np
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# --- KONFIGURASI DARI FEEDBACK ---
M_SAMPLES = 5 # Menggunakan M=5 sudah cukup (Minimum viable) dan menghemat biaya token
MIN_SCORE = 40
MAX_SCORE = 100
SCORE_RANGE = MAX_SCORE - MIN_SCORE
NSV_THRESHOLD = 0.1 # Threshold 10% dari range (setara std dev > 6 poin)

llm_sampler = ChatOpenAI(
    model=os.getenv("LLM_PENILAI", "anthropic/claude-sonnet-4.6"),
    api_key=os.getenv("LLM_BINDING_API_KEY"),
    base_url=os.getenv("LLM_BINDING_HOST"),
    temperature=0.5
)

async def get_sample_score(prompt_text):
    """Fungsi mengambil sampel dengan filter validasi (outlier handling)"""
    try:
        response = await llm_sampler.ainvoke([HumanMessage(content=prompt_text)])
        content = response.content.strip()
        
        # 2. Logika RegEx untuk mengekstrak teks di dalam bracket { ... }
        # Ini akan mengabaikan basa-basi Claude atau markdown ```json
        json_match = re.search(r'\{.*\}', content, re.DOTALL)
        
        if not json_match:
            print(f"⚠️ Gagal menemukan format JSON di output model.")
            return None
            
        clean_json_string = json_match.group(0)
        data = json.loads(clean_json_string)
        
        # Ekstraksi dan Validasi rentang skor
        scores = data.get("scores", {})
        valid_scores = {}
        for k, v in scores.items():
            if isinstance(v, (int, float)) and MIN_SCORE <= v <= MAX_SCORE:
                valid_scores[k] = float(v)
                
        return valid_scores if valid_scores else None
    except Exception as e:
        return None

async def calculate_score_uncertainty(prompt_text):
    print(f"🚀 Memulai {M_SAMPLES} sampling untuk ekstraksi ketidakpastian skor...")
    
    # Menjalankan sampling paralel
    tasks = [get_sample_score(prompt_text) for _ in range(M_SAMPLES)]
    results = await asyncio.gather(*tasks)
    
    # Struktur data penampung
    score_distributions = {
        "n1": [],
        "n2": [],
        "n3": [], 
        "n4": [],
        "n5": []
    }
    
    # Memasukkan skor tervalidasi ke dalam distribusi
    for res in results:
        if not res: continue
        for key in score_distributions.keys():
            if key in res:
                score_distributions[key].append(res[key])
                
    # --- KALKULASI METRIK UNCERTAINTY ---
    evaluation_results = {
        "consensus_scores": {},
        "uncertainty": {"per_criteria": {}}
    }
    
    nsv_dict = {}
    
    for criteria, values in score_distributions.items():
        if not values: continue
        
        mean_score = np.mean(values)
        std_dev = np.std(values, ddof=1) if len(values) > 1 else 0.0
        
        # Metrik 1: NSV (Normalized Score Variance)
        nsv = std_dev / SCORE_RANGE
        nsv_dict[criteria] = nsv
        
        status = "⚠️ PERLU REVIEW" if nsv > NSV_THRESHOLD else "✅ YAKIN"
        
        evaluation_results["consensus_scores"][criteria] = round(mean_score, 2)
        evaluation_results["uncertainty"]["per_criteria"][criteria] = {
            "nsv": round(nsv, 3),
            "std": round(std_dev, 2),
            "status": status,
            "raw_samples": values
        }
        
    # Metrik 2: WAU (Weighted Aggregate Uncertainty)
    # Menghitung agregat dengan bobot n4 dikalikan 2 (Total pembagi 6)
    if all(k in nsv_dict for k in score_distributions.keys()):
        wau = (
            nsv_dict["n1"] + 
            nsv_dict["n2"] + 
            nsv_dict["n3"] + 
            (2 * nsv_dict["n4"]) + 
            nsv_dict["n5"]
        ) / 6
        
        evaluation_results["uncertainty"]["weighted_aggregate"] = round(wau, 3)
        evaluation_results["uncertainty"]["overall_status"] = "⚠️ BUTUH REVIEW HUMAN" if wau > NSV_THRESHOLD else "✅ YAKIN (Konsisten)"
        
        # Mencari kriteria dengan ketidakpastian tertinggi
        evaluation_results["uncertainty"]["most_uncertain_criteria"] = max(nsv_dict, key=nsv_dict.get)

    # --- TAMPILKAN HASIL ---
    print("\n" + "="*60)
    print("📊 HASIL ANALISIS KETIDAKPASTIAN SKOR MAKALAH (NSV & WAU)")
    print("="*60)
    print(json.dumps(evaluation_results, indent=2))
    print("="*60)

# Eksekusi fungsi dengan passing variabel prompt secara eksplisit
# Asumsi: variabel evaluation_prompt sudah berisi prompt lengkap
await calculate_score_uncertainty(evaluation_prompt)

🚀 Memulai 5 sampling untuk ekstraksi ketidakpastian skor...

📊 HASIL ANALISIS KETIDAKPASTIAN SKOR MAKALAH (NSV & WAU)
{
  "consensus_scores": {
    "n1": 78.0,
    "n2": 72.0,
    "n3": 68.0,
    "n4": 63.2,
    "n5": 70.4
  },
  "uncertainty": {
    "per_criteria": {
      "n1": {
        "nsv": 0.0,
        "std": 0.0,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          78.0,
          78.0,
          78.0,
          78.0,
          78.0
        ]
      },
      "n2": {
        "nsv": 0.0,
        "std": 0.0,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          72.0,
          72.0,
          72.0,
          72.0,
          72.0
        ]
      },
      "n3": {
        "nsv": 0.0,
        "std": 0.0,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          68.0,
          68.0,
          68.0,
          68.0,
          68.0
        ]
      },
      "n4": {
        "nsv": 0.027,
        "std": 1.64,
        "status": "\u2705 YAKIN",
       

### Run ke 2 end

In [ ]:
# --- KONFIGURASI DARI FEEDBACK ---
M_SAMPLES = 5 # Menggunakan M=5 sudah cukup (Minimum viable) dan menghemat biaya token
MIN_SCORE = 40
MAX_SCORE = 100
SCORE_RANGE = MAX_SCORE - MIN_SCORE
NSV_THRESHOLD = 0.1 # Threshold 10% dari range (setara std dev > 6 poin)

llm_sampler = ChatOpenAI(
    model=os.getenv("LLM_PENILAI", "anthropic/claude-sonnet-4.6"),
    api_key=os.getenv("LLM_BINDING_API_KEY"),
    base_url=os.getenv("LLM_BINDING_HOST"),
    temperature=1.0
)


await calculate_score_uncertainty(evaluation_prompt)

🚀 Memulai 5 sampling untuk ekstraksi ketidakpastian skor...

📊 HASIL ANALISIS KETIDAKPASTIAN SKOR MAKALAH (NSV & WAU)
{
  "consensus_scores": {
    "n1": 81.2,
    "n2": 74.0,
    "n3": 69.6,
    "n4": 66.2,
    "n5": 72.0
  },
  "uncertainty": {
    "per_criteria": {
      "n1": {
        "nsv": 0.03,
        "std": 1.79,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          78.0,
          82.0,
          82.0,
          82.0,
          82.0
        ]
      },
      "n2": {
        "nsv": 0.031,
        "std": 1.87,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          72.0,
          75.0,
          76.0,
          72.0,
          75.0
        ]
      },
      "n3": {
        "nsv": 0.028,
        "std": 1.67,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          68.0,
          70.0,
          70.0,
          68.0,
          72.0
        ]
      },
      "n4": {
        "nsv": 0.027,
        "std": 1.64,
        "status": "\u2705 YAKIN",

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


M_SAMPLES = 5 # Menggunakan M=5 sudah cukup (Minimum viable) dan menghemat biaya token
MIN_SCORE = 40
MAX_SCORE = 100
SCORE_RANGE = MAX_SCORE - MIN_SCORE
NSV_THRESHOLD = 0.1 # Threshold 10% dari range (setara std dev > 6 poin)

llm_sampler = ChatOpenAI(
    model=os.getenv("LLM_PENILAI", "anthropic/claude-sonnet-4.6"),
    api_key=os.getenv("LLM_BINDING_API_KEY"),
    base_url=os.getenv("LLM_BINDING_HOST"),
    temperature=1.5
)

await calculate_score_uncertainty(evaluation_prompt)

🚀 Memulai 5 sampling untuk ekstraksi ketidakpastian skor...

📊 HASIL ANALISIS KETIDAKPASTIAN SKOR MAKALAH (NSV & WAU)
{
  "consensus_scores": {
    "n1": 79.6,
    "n2": 73.2,
    "n3": 68.8,
    "n4": 65.6,
    "n5": 71.6
  },
  "uncertainty": {
    "per_criteria": {
      "n1": {
        "nsv": 0.037,
        "std": 2.19,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          82.0,
          82.0,
          78.0,
          78.0,
          78.0
        ]
      },
      "n2": {
        "nsv": 0.027,
        "std": 1.64,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          75.0,
          75.0,
          72.0,
          72.0,
          72.0
        ]
      },
      "n3": {
        "nsv": 0.018,
        "std": 1.1,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          70.0,
          70.0,
          68.0,
          68.0,
          68.0
        ]
      },
      "n4": {
        "nsv": 0.022,
        "std": 1.34,
        "status": "\u2705 YAKIN",

### Veramika G

In [20]:
from src.prompt.prompts import PROMPT_PENILAIAN

# Veramika
from src.helper.helper import _read_pdf
makalah_path = "../data/Makalah Uji/Makalah Bidang Veramika Ginting.pdf"


makalah_text = _read_pdf(makalah_path)
print(makalah_text)



evaluation_prompt_veramika = PROMPT_PENILAIAN.format(
    assessment_context=context_response_llm,
    makalah_text=makalah_text,
    tema_text="""Strategi Balai Besar POM Dalam Meningkatkan Perkuatan Pengawasan 
Obat dan Makanan Berbasis Resiko yang Adatif, Responsive dan Terintegrasi 
di era efisiensi untuk mendukung Asta Cita Pemerintah, daya saing dan 
peningkatan perekonomian dengan memanfaatakan Teknologi Informasi 
Digitalisasi.""", 
)

eval_response = LLM_PENILAI.invoke(evaluation_prompt_veramika)
eval_response


BADAN POM
STRATEGI ADAPTIF DALAM
MEWUJUDKAN ASTA CITA DAN
PENGUATAN EKONOMI
NASIONAL MELALUI
TRANSFORMASI DIGITAL DAN
PENGAWASAN BERBASIS
RESIKOOLEH BALAI BESAR
POM
Makalah Bidang oleh Veramika Ginting, S.Si., Apt.,M.H.
Veramika Ginting
14 Maret 2026
STRATEGI ADAPTIF DALAM MEWUJUDKAN ASTA CITA DAN
PENGUATAN EKONOMI NASIONAL MELALUI TRANSFORMASI DIGITAL
DAN PENGAWASAN BERBASIS RESIKO OLEH BALAI BESAR POM
I. PENDAHULUAN
A. Latar Belakang
Badan Pengawas Obat dan Makanan (BPOM) memiliki peran penting
dalam memastikan bahwa semua produk Obat (Obat, Obat Tradisional, Kosmetik
dan Suplemen Kesehatan) dan Makanan (Pangan Olahan) yang beredar aman ,
berkhasiat/bermanfaat ketika dikonsumsi oleh masyarakat. Tugas, fungsi dan
kewenangan Badan POM diatur dalam Peraturan Presiden Nomor 80 Tahun 2017
Tentang Badan Pengawas Obat dan Makanan. Selanjutnya untuk Unit Pelaksana
Teknis yaitu Balai Besar/Balai POM dan Loka POM berdasarkan Keputusan
Kepala Badan Pengawas Obat dan Makanan Nomor : 05018/KBPOM/

AIMessage(content='```json\n{\n  "Ringkasan": "Makalah ini berjudul \'Strategi Adaptif dalam Mewujudkan Asta Cita dan Penguatan Ekonomi Nasional melalui Transformasi Digital dan Pengawasan Berbasis Risiko oleh Balai Besar POM\', ditulis oleh Veramika Ginting. Makalah membahas peran strategis Balai Besar POM sebagai Unit Pelaksana Teknis (UPT) BPOM dalam mendukung program prioritas pemerintah (Asta Cita), khususnya program Makan Bergizi Gratis dan penuntasan TBC. Struktur makalah mencakup pendahuluan (latar belakang, rumusan masalah, dan tujuan), pembahasan (analisis SWOT, konsep transformasi digital, dampak yang diharapkan, serta rencana aksi jangka pendek, menengah, dan panjang), serta kesimpulan. Makalah menekankan tiga pilar utama: (1) penyelarasan strategi Balai Besar POM dengan Asta Cita, (2) transformasi digital sebagai akselerator efisiensi dan transparansi birokrasi, dan (3) penerapan pengawasan berbasis risiko untuk mengoptimalkan sumber daya di tengah keterbatasan anggaran. M

In [21]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


M_SAMPLES = 5
MIN_SCORE = 40
MAX_SCORE = 100
SCORE_RANGE = MAX_SCORE - MIN_SCORE
NSV_THRESHOLD = 0.1 # Threshold 10% dari range (setara std dev > 6 poin)

llm_sampler = ChatOpenAI(
    model=os.getenv("LLM_PENILAI", "anthropic/claude-sonnet-4.6"),
    api_key=os.getenv("LLM_BINDING_API_KEY"),
    base_url=os.getenv("LLM_BINDING_HOST"),
    temperature=0.5
)

await calculate_score_uncertainty(evaluation_prompt_veramika)

🚀 Memulai 5 sampling untuk ekstraksi ketidakpastian skor...

📊 HASIL ANALISIS KETIDAKPASTIAN SKOR MAKALAH (NSV & WAU)
{
  "consensus_scores": {
    "n1": 82.0,
    "n2": 78.0,
    "n3": 74.4,
    "n4": 71.6,
    "n5": 70.4
  },
  "uncertainty": {
    "per_criteria": {
      "n1": {
        "nsv": 0.0,
        "std": 0.0,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          82.0,
          82.0,
          82.0,
          82.0,
          82.0
        ]
      },
      "n2": {
        "nsv": 0.0,
        "std": 0.0,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          78.0,
          78.0,
          78.0,
          78.0,
          78.0
        ]
      },
      "n3": {
        "nsv": 0.022,
        "std": 1.34,
        "status": "\u2705 YAKIN",
        "raw_samples": [
          75.0,
          75.0,
          75.0,
          72.0,
          75.0
        ]
      },
      "n4": {
        "nsv": 0.015,
        "std": 0.89,
        "status": "\u2705 YAKIN",
    

#### Naikin temperature ke 1.0


In [25]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


M_SAMPLES = 5
MIN_SCORE = 40
MAX_SCORE = 100
SCORE_RANGE = MAX_SCORE - MIN_SCORE
NSV_THRESHOLD = 0.1 # Threshold 10% dari range (setara std dev > 6 poin)

llm_sampler = ChatOpenAI(
    model=os.getenv("LLM_PENILAI", "anthropic/claude-sonnet-4.6"),
    api_key=os.getenv("LLM_BINDING_API_KEY"),
    base_url=os.getenv("LLM_BINDING_HOST"),
    temperature=1.0
)

await calculate_score_uncertainty(evaluation_prompt_veramika)

🚀 Memulai 5 sampling untuk ekstraksi ketidakpastian skor...

📊 HASIL ANALISIS KETIDAKPASTIAN SKOR MAKALAH (NSV & WAU)
{
  "consensus_scores": {},
  "uncertainty": {
    "per_criteria": {}
  }
}


In [26]:
llm_sampler.ainvoke(HumanMessage(content=evaluation_prompt_veramika))

<coroutine object BaseChatModel.ainvoke at 0x000001A4DCFD8270>

In [27]:
llm_sampler

ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x000001A4DB7034F0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001A4DBE09B70>, root_client=<openai.OpenAI object at 0x000001A4DB7015D0>, root_async_client=<openai.AsyncOpenAI object at 0x000001A4DBE097E0>, model_name='anthropic/claude-sonnet-4.6', temperature=1.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://openrouter.ai/api/v1', openai_proxy=None, stream_chunk_timeout=120.0)

In [28]:
LLM_PENILAI.ainvoke(HumanMessage(content=evaluation_prompt_veramika))

<coroutine object BaseChatModel.ainvoke at 0x000001A4DCFDB060>

In [55]:
evaluation_prompt

'\n---Role---\n\nAnda adalah evaluator akademik sebagai Panitia Seleksi yang bertugas menilai kualitas substansi makalah secara objektif dan sistematis. Penilaian harus didasarkan hanya pada isi makalah yang tersedia, dengan mempertimbangkan konteks jabatan yang dituju.\n\n---Goal---\n\nMelakukan penilaian terhadap makalah berdasarkan kriteria penilaian yang telah ditentukan, memberikan skor numerik untuk setiap kriteria.\n\n---Konteks Jabatan---\n\n## 1. Profil Jabatan\nJabatan `{selected_jabatan}` merupakan posisi kunci dalam struktur organisasi BPOM yang bertanggung jawab atas pengawasan dan regulasi produk obat dan makanan. Tugas utamanya meliputi perencanaan, pelaksanaan, dan evaluasi fungsi pengawasan serta penyusunan kebijakan yang mendukung layanan publik yang aman.\n\n## 2. Kompetensi yang Dipersyaratkan\n### Kompetensi Teknis\n- Memiliki keahlian dalam pengawasan dan evaluasi produk obat dan makanan.\n- Menguasai prosedur pengujian laboratorium serta penegakan hukum dalam bid